# Data Collection – Web Scraping

**Objective:** Scrape the Wikipedia page *"List of Falcon 9 and Falcon Heavy launches"* to build a second, independent launch-records dataset for cross-checking the API data.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`

## Workflow (flowchart)
```
[GET Wikipedia page HTML]
        |
        v
[Parse with BeautifulSoup -> find launch-records <table>]
        |
        v
[Extract column headers]
        |
        v
[Loop table rows -> extract Date, Site, Payload, Orbit,
 Customer, Outcome, Booster version, etc.]
        |
        v
[Assemble DataFrame -> clean -> dataset_part_2.csv]
```


In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re


### Step 1: Fetch the Wikipedia page HTML

In [2]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"


headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

response = requests.get(static_url, headers=headers)
print(response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    print(soup.title.string)
else:
    print(f"Failed to fetch page, status code: {response.status_code}")


200
List of Falcon 9 and Falcon Heavy launches - Wikipedia


### Step 2: Locate the launch-records tables

In [3]:
html_tables = soup.find_all('table', class_='wikitable plainrowheaders collapsible')
print("Number of candidate tables found:", len(html_tables))

first_launch_table = html_tables[2]


Number of candidate tables found: 9


In [4]:
def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not colunm_name.strip().isdigit():
        colunm_name = colunm_name.strip()
        return colunm_name

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)


['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### Step 3: Parse every row of every launch table into a dictionary

In [5]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []


In [6]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

import unicodedata

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', class_='wikitable plainrowheaders collapsible')):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            launch_site = row[2].a.string if row[2].a else None
            launch_dict['Launch site'].append(launch_site)

            payload = row[3].a.string if row[3].a else None
            launch_dict['Payload'].append(payload)

            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            orbit = row[5].a.string if row[5].a else None
            launch_dict['Orbit'].append(orbit)

            customer = row[6].a.string if row[6].a else None
            launch_dict['Customer'].append(customer)

            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print("Rows extracted:", extracted_row)


Rows extracted: 121


### Step 4: Build the DataFrame and export

In [7]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.head()


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


In [8]:
df.to_csv('dataset_part_2.csv', index=False)
print("Saved dataset_part_2.csv with shape:", df.shape)


Saved dataset_part_2.csv with shape: (121, 11)


## Summary
- Scraped the Wikipedia *"List of Falcon 9 and Falcon Heavy launches"* page with `requests` + `BeautifulSoup`.
- Parsed table headers programmatically and iterated every `<tr>`/`<td>` to extract flight number, date, booster version, launch site, payload, payload mass, orbit, customer, outcome and booster-landing status.
- Exported the result as `dataset_part_2.csv`, an independent record set used to cross-validate the API-sourced data.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`
